# 03 — Validation split (28-day holdout) + popularity baseline

**Split:**
- `train_pos`: events_positive với `date < VALID_START` (2026-03-13)
- `valid_pos`: events_positive với `date ∈ [VALID_START, TRAIN_END]`
- `valid_users`: user có trong cả 2 (mô phỏng test_users là user có history)

**Metric:** `recall_at_k`, `ndcg_at_k` từ `utils.metrics`.

**Baseline:** top-10 item theo `contacts_total` (tier A only).
Mục tiêu sau ranker: Recall@10 ≥ 2x baseline.

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local mode: no GCS egress guardrail needed.
print("Local kernel: reading from CACHE_DIR.")

In [ ]:
import time
import pandas as pd
import numpy as np

from utils.metrics import mean_recall_at_k, mean_ndcg_at_k

t0 = time.time()
evt = pd.read_parquet(
    f"{CACHE_DIR}/events_positive.parquet",
    columns=["user_id", "item_id", "event_type", "date"],
)
evt["date"] = pd.to_datetime(evt["date"])
print(f"events_positive: {len(evt):,} | {time.time()-t0:.1f}s")

VALID_START_TS = pd.Timestamp(VALID_START)
TRAIN_END_TS = pd.Timestamp(TRAIN_DATE_END)
train_mask = evt["date"] < VALID_START_TS
valid_mask = (evt["date"] >= VALID_START_TS) & (evt["date"] <= TRAIN_END_TS)
train_pos = evt[train_mask]
valid_pos = evt[valid_mask]
print(f"train_pos: {len(train_pos):,} ({len(train_pos)/len(evt)*100:.1f}%)")
print(f"valid_pos: {len(valid_pos):,} ({len(valid_pos)/len(evt)*100:.1f}%)")

In [ ]:
# valid_users = unique user trong cả train_pos và valid_pos
train_users = set(train_pos["user_id"].unique())
valid_users_full = set(valid_pos["user_id"].unique())
valid_users = train_users & valid_users_full
print(f"|train users|: {len(train_users):,}")
print(f"|valid users (any)|: {len(valid_users_full):,}")
print(f"|valid_users (∩ train)|: {len(valid_users):,}")

# Ground-truth: user_id -> set(item_id) in valid window
gt = (valid_pos[valid_pos["user_id"].isin(valid_users)]
      .groupby("user_id")["item_id"]
      .agg(set)
      .to_dict())
print(f"GT users: {len(gt):,} | mean items/user: "
      f"{np.mean([len(v) for v in gt.values()]):.2f}")

In [ ]:
# Save split for downstream notebooks
train_pos.to_parquet(f"{CACHE_DIR}/train_pos.parquet", index=False)
valid_pos.to_parquet(f"{CACHE_DIR}/valid_pos.parquet", index=False)

# Save valid_users + gt as parquet (gt as long form)
gt_long = (valid_pos[valid_pos["user_id"].isin(valid_users)]
           [["user_id", "item_id"]].drop_duplicates())
gt_long.to_parquet(f"{CACHE_DIR}/valid_gt.parquet", index=False)
print(f"valid_gt: {len(gt_long):,} pairs")

In [ ]:
# ---- Popularity baseline: tier-A items by contacts_total, top-10 ----
enr = pd.read_parquet(
    f"{CACHE_DIR}/dim_listing_enriched.parquet",
    columns=["item_id", "tier", "contacts_total", "n_pos_train"],
)
pop_pool = enr[enr["tier"] == "A"].sort_values(
    ["contacts_total", "n_pos_train"], ascending=False
)
top10_global = pop_pool["item_id"].head(10).tolist()
print("Top-10 popularity:", top10_global[:5], "...")

# Predict same top-10 for every valid user
preds_baseline = {u: top10_global for u in gt.keys()}
r10 = mean_recall_at_k(preds_baseline, gt, k=10)
n10 = mean_ndcg_at_k(preds_baseline, gt, k=10)
print(f"\nBaseline Recall@10: {r10:.4f}")
print(f"Baseline NDCG@10  : {n10:.4f}")
print("(Mục tiêu sau ranker: Recall@10 ≥ 2x baseline)")

In [ ]:
# Save baseline preds for sanity comparison later
import json
with open(f"{CACHE_DIR}/baseline_preds.json", "w") as f:
    json.dump({"top10_global": top10_global,
               "recall_at_10": r10, "ndcg_at_10": n10}, f, indent=2)
print("Baseline saved.")